# Agent Identity (Governance)

In [10]:
# @title Installs
############################################

# Google Cloud Vertex AI (Agent Engine) & Google ADK installieren
# Feste Versionen statt --upgrade: Der Deploy pinnt dieselben Versionen, damit das Pickle in der Cloud passt
%pip install --quiet "google-cloud-aiplatform[agent_engines,adk]==2.6.0" "google-adk==2.13.0" "google-genai==2.33.0"

# Versionen kurz prüfen
%pip show google-adk google-cloud-aiplatform

Note: you may need to restart the kernel to use updated packages.
Name: google-adk
Version: 2.13.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: aiosqlite, authlib, click, cryptography, fastapi, google-auth, google-genai, graphviz, httpx, jsonschema, opentelemetry-api, opentelemetry-sdk, packaging, pydantic, python-dotenv, python-multipart, pyyaml, requests, starlette, tenacity, typing-extensions, uvicorn, watchdog, websockets
Required-by: 
---
Name: google-cloud-aiplatform
Version: 2.6.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: certifi, docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cl

In [11]:
# @title Imports
############################################

import logging
import os
import warnings
from importlib.metadata import version

import vertexai
from google.adk.agents import LlmAgent
from google.adk.integrations.agent_registry import AgentRegistry
from google.cloud import bigquery
from google.genai.errors import ClientError
from vertexai import types
from vertexai.agent_engines import AdkApp

warnings.filterwarnings('ignore')

print("✅ Alle Imports erfolgreich geladen!")

✅ Alle Imports erfolgreich geladen!


In [12]:
# @title Environmental Variables & GCP Setup
############################################

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"  # Agent Engine Region
STAGING_BUCKET = "gs://deltorobarba-agent-staging"
GCS_DIR = "agent_governance"  # eigener Ordner im Bucket: agents.ipynb benutzt 'agent_engine/'
MODEL = "gemini-3.8-flash"

# Nach dem ersten Deploy hier eintragen, damit ein zweiter Lauf keine weitere Engine anlegt
RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888"

# These tell the underlying google-genai client to use Vertex AI (Gemini 3.8 Flash uses 'global')
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# Initialize Vertex AI SDK & Client (v1beta1: nötig für identity_type und agent_gateway_config)
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION, http_options=dict(api_version="v1beta1"))

print("✅ Erfolgreich über Application Default Credentials (ADC) verbunden.")

✅ Erfolgreich über Application Default Credentials (ADC) verbunden.


**Agent Identity I: Single Agent ohne Tools (Least Privilege)**

Ein einzelner Agent wird auf Agent Runtime mit eigener **Agent Identity** deployt und direkt abgefragt, zuerst ohne Tools. In **UC4** (unten) bekommt er ein BigQuery-Tool, dessen Recht nur er hat.

```mermaid
%%{init: {'themeVariables': {'fontSize': '13px'}, 'flowchart': {'nodeSpacing': 20, 'rankSpacing': 28, 'padding': 6}}}%%
flowchart LR
    U(["Du<br/>Access-Token"]) -->|"query"| A["agent_identity<br/>Agent Identity (SPIFFE)"]
    A -->|"eigene Identität"| G[("Gemini")]
    A -.- R["Agent Registry"]
    A -->|"Tool, eigene Identität (UC4)"| B[("BigQuery")]
```

* **Was:** Mit `identity_type=AGENT_IDENTITY` bekommt die Engine eine eigene, von Google verwaltete Identität statt des Reasoning-Engine-Service-Agents: eine SPIFFE-ID wie `agents.global.proj-<Projektnummer>.system.id.goog/resources/aiplatform/projects/<Nr>/locations/<Region>/reasoningEngines/<ID>`. Gehört das Projekt zu einer Organisation, steht dort `org-<Org-ID>` statt `proj-<Projektnummer>`.
* **Kein Schlüssel:** Es gibt keinen Service Account und keinen Key. Das Token ist an ein Zertifikat der Runtime gebunden und gilt nur dort.
* **IAM:** Rollen gehen direkt an den Principal `principal://<SPIFFE-ID>`. Beim Erstellen gibt Google allen Agent Identities des Projekts `roles/aiplatform.agentDefaultAccess` (Modell, Sessions, Memory, Logging); für den Agent ohne Tools reicht das, das Tool in UC4 braucht ein eigenes Recht.
* **Nur beim `create`:** Die Identität lässt sich nicht per `update` einschalten. Wird die Engine gelöscht und neu erstellt, entsteht eine neue Identität, und alte Rollen gelten für sie nicht.
* **Voraussetzung für Agent Gateway** (`gateway.ipynb`, UC5).

Summary: Die Engine „Agent Identity“ läuft auf Agent Runtime und tritt auf als

```
principal://agents.global.proj-622694106880.system.id.goog/…/reasoningEngines/3345367756479397888
```
* Es gibt keinen Service Account und keinen Key, das Token gilt nur innerhalb der Runtime.
* „Isoliert“ heißt hier: Diese Identität hat nur dieser eine Agent. Den Unterschied sieht man an deinen anderen beiden Engines, Pipeline und MCP-Analyst. Beide laufen als derselbe Service Agent `service-622694106880@gcp-sa-aiplatform-re…`. Ein Beispiel: Das `run.invoker`-Recht, das wir dem MCP-Analysten auf den BigQuery-MCP-Server gegeben haben, hat damit automatisch auch die Pipeline. Rollen an die `principal://…`-ID gelten dagegen nur für diesen Agent.
* Eine Einschränkung gibt es: `agentDefaultAccess` vergibt Google an *alle* Agent Identities des Projekts (`principalSet://…`). Die drei Rollen aus der optionalen IAM-Zelle sind für diesen Agent überflüssig.

In [4]:
# @title Define Agent
############################################

# Ein einzelner Agent ohne Tools: braucht außer den Standardrechten seiner Agent Identity nichts
root_agent = LlmAgent(
    name="agent_identity",
    model=MODEL,
    description="Minimal agent to test Agent Identity, Agent Registry and Agent Gateway.",
    instruction="You are a concise assistant. Answer in at most three sentences.",
)

print(f"✅ Agent '{root_agent.name}' definiert.")

✅ Agent 'agent_identity' definiert.


In [5]:
# @title Local Test (Before Deploying to Agent Engine)
############################################
# Läuft mit DEINEN Credentials: prüft den Agent, noch nicht seine Identität

local_app = AdkApp(agent=root_agent)


async def run_local_test():
    user_id = "deltorobarba"
    session = await local_app.async_create_session(user_id=user_id)
    print(f"🧪 Local Test Session ID: {session['id']}")

    async for event in local_app.async_stream_query(
        user_id=user_id,
        session_id=session["id"],
        message="Explain in two sentences what a SPIFFE ID is.",
    ):
        for part in event.get("content", {}).get("parts", []):
            if "text" in part:
                print(f"\n--- Local Output ({event.get('author')}) ---\n{part['text']}")

await run_local_test()

🧪 Local Test Session ID: 5ca32e9f-f8ce-4264-83f6-2b0eebe1b4ae

--- Local Output (agent_identity) ---
A SPIFFE ID is a standardized URI format used to uniquely identify a workload across dynamic, heterogeneous environments. It is typically embedded within a cryptographic document, like an X.509 certificate, to enable secure mutual authentication between services.


In [6]:
# @title Deploy Agent with Agent Identity on Agent Engine (⚠️ --> once only!)
############################################

"""
Agent Identity: https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-identity
identity_type=AGENT_IDENTITY geht nur beim create; danach RESOURCE_NAME oben eintragen.
"""

# Fortschritt anzeigen: das SDK loggt LRO und Resource-Namen auf INFO, was Jupyter sonst verschluckt
_log = logging.getLogger("vertexai_genai.agentengines")
_log.setLevel(logging.INFO)
_log.propagate = False
if not _log.handlers:
    _log.addHandler(logging.StreamHandler())

DEPLOY_CONFIG = {
    "display_name": "Agent Identity",
    "description": "Single agent with Agent Identity",
    "staging_bucket": STAGING_BUCKET,
    "gcs_dir_name": f"{GCS_DIR}/agent_identity",  # eigener Ordner: das Pickle von agents.ipynb bleibt unberührt
    # Same versions as locally: the agent is pickled here and unpickled in the cloud
    "requirements": [
        f"google-cloud-aiplatform[agent_engines,adk]=={version('google-cloud-aiplatform')}",
        f"google-adk=={version('google-adk')}",
        f"google-genai=={version('google-genai')}",
        f"cloudpickle=={version('cloudpickle')}",
        f"pydantic=={version('pydantic')}",
    ],
    "env_vars": {"GOOGLE_CLOUD_LOCATION": "global"},  # Gemini 3.8 Flash läuft auf 'global'
}

if RESOURCE_NAME:
    print("Engine existiert schon:", RESOURCE_NAME)
else:
    remote_agent = client.agent_engines.create(
        agent=AdkApp(agent=root_agent),
        config={**DEPLOY_CONFIG, "identity_type": types.IdentityType.AGENT_IDENTITY},
    )
    RESOURCE_NAME = remote_agent.api_resource.name
    print(f"\n✅ Engine angelegt: {RESOURCE_NAME}")
    print("   👉 oben in 'Environmental Variables & GCP Setup' als RESOURCE_NAME eintragen")

AGENT_IDENTITY = client.agent_engines.get(name=RESOURCE_NAME).api_resource.spec.effective_identity
AGENT_PRINCIPAL = f"principal://{AGENT_IDENTITY}"
print("SPIFFE-ID: ", AGENT_IDENTITY)
print("IAM-Member:", AGENT_PRINCIPAL)

Engine existiert schon: projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888
SPIFFE-ID:  agents.global.proj-622694106880.system.id.goog/resources/aiplatform/projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888
IAM-Member: principal://agents.global.proj-622694106880.system.id.goog/resources/aiplatform/projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888


In [7]:
# @title (Optional) IAM-Rollen für die Agent Identity
############################################
# Für diesen Agent nicht nötig (Standardrechte, siehe oben). Die Google-Doku empfiehlt für Agents diese
# Rollen; Agents mit Tools brauchen zusätzlich die Rollen ihrer Zielsysteme (z. B. discoveryengine.viewer).

AGENT_ROLES = [
    "roles/aiplatform.expressUser",             # Inferenz, Sessions, Memory
    "roles/serviceusage.serviceUsageConsumer",  # Kontingent des Projekts nutzen
    "roles/browser",                            # Grundfunktionen von Google Cloud
]
for role in AGENT_ROLES:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} --member="{AGENT_PRINCIPAL}" --role="{role}" --condition=None --quiet > /dev/null && echo "✅ {role}"
print("IAM-Propagation dauert einige Minuten.")

Updated IAM policy for project [deltorobarba].
✅ roles/aiplatform.expressUser
Updated IAM policy for project [deltorobarba].
✅ roles/serviceusage.serviceUsageConsumer
Updated IAM policy for project [deltorobarba].
✅ roles/browser
IAM-Propagation dauert einige Minuten.


In [8]:
# @title Test Agent ====> Single Turn Query
############################################
# Du rufst mit deinem Access-Token auf; der Agent legt die Session an und ruft Gemini mit SEINER Identität

remote_app = client.agent_engines.get(name=RESOURCE_NAME)


async def run_remote_test():
    user_id = "deltorobarba"
    print(f"Connecting to: {RESOURCE_NAME}...")
    try:
        session = await remote_app.async_create_session(user_id=user_id)
        print(f"✅ Managed Session ID: {session['id']}")

        async for event in remote_app.async_stream_query(
            user_id=user_id,
            session_id=session["id"],
            message="Explain in two sentences what a SPIFFE ID is.",
        ):
            if event.get("error_message"):
                print(f"❌ [{event.get('author')}] {event.get('error_code')}: {event['error_message']}")
            for part in event.get("content", {}).get("parts", []):
                if "text" in part:
                    print(f"\n--- Agent Output ({event.get('author')}) ---\n{part['text']}")
    except ClientError as e:
        print("❌", e.message)
        print("\n-> siehe Troubleshooting am Ende")

await run_remote_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888...
✅ Managed Session ID: 3229233112105877504

--- Agent Output (agent_identity) ---
A SPIFFE ID is a standardized Uniform Resource Identifier (URI) formatted as `spiffe://<trust-domain>/<workload-path>` that uniquely identifies a software workload in a distributed environment. It decouples workload identity from physical network infrastructure, enabling secure, cryptographic mutual authentication across heterogeneous platforms.


In [ ]:
# Check Agent Registry: Eintrag und Laufzeit-Identität
############################################
# Der Eintrag erscheint wenige Minuten nach dem Deploy

agent_registry = AgentRegistry(project_id=PROJECT_ID, location=LOCATION)
RUNTIME_KEY = "agentregistry.googleapis.com/system/RuntimeReference"
IDENTITY_KEY = "agentregistry.googleapis.com/system/RuntimeIdentity"

registered = agent_registry.list_agents().get("agents", [])
print("Alle registrierten Agents mit Laufzeit-Identität:")
for a in registered:
    principal = a.get("attributes", {}).get(IDENTITY_KEY, {}).get("principal", "–")
    print(f"  {a.get('displayName', '?'):22} {principal}")

engine_id = RESOURCE_NAME.split("/")[-1]
entry = next((a for a in registered
              if a.get("attributes", {}).get(RUNTIME_KEY, {}).get("uri", "").endswith(engine_id)), None)
if entry is None:
    print("\nDieser Agent ist noch nicht in der Registry – in ein paar Minuten erneut ausführen.")
else:
    principal = entry["attributes"].get(IDENTITY_KEY, {}).get("principal")
    print("\nDieser Agent:")
    print("  Registry-Eintrag:  ", entry["name"])
    print("  Protokolle:        ", [p.get("type") for p in entry.get("protocols", [])])
    print("  Laufzeit-Identität:", principal)
    print("  = SPIFFE-ID der Engine:", principal == AGENT_PRINCIPAL)

Alle registrierten Agents mit Laufzeit-Identität:
  Workspace Agent        –
  Agent Identity         principal://agents.global.proj-622694106880.system.id.goog/resources/aiplatform/projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888
  MCP BQ Analyst         sa://service-622694106880@gcp-sa-aiplatform-re.iam.gserviceaccount.com
  Agent Pipeline         sa://service-622694106880@gcp-sa-aiplatform-re.iam.gserviceaccount.com

Dieser Agent:
  Registry-Eintrag:   projects/deltorobarba/locations/us-central1/agents/agentregistry-00000000-0000-0000-9bd5-c1183342042b
  Protokolle:         ['CUSTOM']
  Laufzeit-Identität: principal://agents.global.proj-622694106880.system.id.goog/resources/aiplatform/projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888
  = SPIFFE-ID der Engine: True


**Agent Identity II: Tool mit Rechten nur für diesen Agent (Least Privilege)**

Bis hier konnte der Agent nur mit Gemini sprechen, dafür reichen die Standardrechte. Jetzt bekommt er ein Tool, das eine Google-API aufruft, und das Recht dafür bekommt **nur seine Agent Identity**. Das ist der eigentliche Nutzen von Agent Identity.

```
Du ──▶ Agent Runtime: Agent "agent_identity"        läuft als principal://agents.global.proj-622694106880…/reasoningEngines/3345…
          │  Tool query_public_dataset               Python-Funktion im Agent, ADC der Runtime = die Agent Identity
          ▼
       BigQuery (bigquery.googleapis.com)            prüft: hat genau dieser Principal roles/bigquery.jobUser?
          │
          ▼
       bigquery-public-data.usa_names
```

**Vergleich mit UC1 (`mcp.ipynb`)**

| | UC1: Agent mit MCP-Server | UC4: Agent mit eigener Identität |
|---|---|---|
| Wer fragt BigQuery ab | das Dienstkonto des MCP-Servers `bq-mcp-analyst@…`, egal welcher Agent den Server aufruft | der Agent selbst, als `principal://…` |
| Als wer läuft der Agent | Reasoning-Engine-Service-Agent `service-622694106880@gcp-sa-aiplatform-re…`, **geteilt** von allen Engines ohne Agent Identity (Pipeline, MCP-Analyst) | eigene Agent Identity, nur diese eine Engine |
| Wem gibt man das Recht | `run.invoker` an den Service Agent: gilt damit auch für die Pipeline | `bigquery.jobUser` an den Principal: gilt nur für diesen Agent |
| Was BigQuery als Aufrufer speichert | `serviceAccount:bq-mcp-analyst@…` | `principal://…/reasoningEngines/3345…` (Audit in UC4g) |

**Ablauf:** UC4a Tool und Agent → UC4b lokal (als du) → UC4c deployen → UC4d Test **ohne** Recht (erwartet: 403 `bigquery.jobs.create`) → UC4e Recht nur für den Principal → UC4f Test **mit** Recht → UC4g Audit: wer hat BigQuery abgefragt.

**Ergebnis (2026-10-26):** UC4d: Das Tool bekam `403 … bigquery.jobs.create`, der Agent hat den Fehler gemeldet und keine Zahlen genannt. UC4f, zwei Minuten nach UC4e: echte Zahlen (Jennifer 58,385 · Amanda 35,818 · Jessica 33,920). UC4g: BigQuery führt den Job unter `principal://iam.googleapis.com/projects/622694106880/locations/global/workloadIdentityPools/agents.global.proj-622694106880.system.id.goog/subject/resources/aiplatform/projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888`, den lokalen Test unter `user:alexmoheit@gmail.com`. Nebenbei: Die Runtime ruft BigQuery über `bigquery.mtls.googleapis.com` auf, also mit dem Zertifikat, an das das Token der Agent Identity gebunden ist.

**Warum nicht der MCP-Server aus UC1?** (getestet am 2026-10-26)

* In der Runtime liefert `fetch_id_token()` ein ID-Token **der Agent Identity**: `sub = spiffe://agents.global.proj-622694106880…/reasoningEngines/3345…`, ausgestellt von Googles STS (`iss = https://sts.googleapis.com/…/workloadIdentityPools/agents.global.proj-622694106880.system.id.goog`), ohne `email`.
* Cloud Run nimmt dieses Token nicht an: **401** „The access token could not be verified“, auch nachdem der Principal `roles/run.invoker` auf dem Dienst hatte. Das Binding selbst ließ sich setzen; die Anmeldung scheitert vorher. Cloud Run erwartet ID-Tokens von `accounts.google.com` (Nutzer oder Dienstkonto).
* Folge: Mit Agent Identity spricht der Agent Google-APIs an (Access-Token aus ADC, wie hier BigQuery), aber nicht einen privaten Cloud-Run-Dienst per ID-Token. Für MCP mit Agent Identity bleiben die von Google verwalteten MCP-Server (`*.googleapis.com`), später zu prüfen.

**Warnung aus demselben Test:** Scheitert die Verbindung eines `McpToolset`, verschwinden seine Tools still aus dem Agent. Das Modell sieht dann gar kein Tool, die Regel „bei Tool-Fehlern keine Zahlen“ greift nicht, und einmal hat es Zahlen erfunden (Jennifer 58,379 statt 58,385). Ein Funktions-Tool wie hier ist immer da und gibt Fehler als Text zurück; dann greift die Regel.

In [ ]:
# Tool und Agent definieren
############################################
# Das Tool ist eine normale Python-Funktion. Sie läuft dort, wo der Agent läuft: lokal als du (ADC deines
# Users), deployed als Agent Identity (ADC der Runtime liefert dort automatisch deren Token).
# Den BigQuery-Client baut sie erst beim Aufruf: So landen keine Credentials und keine Verbindung im Pickle.

ANALYST_INSTRUCTION = """
You are a data analyst with access to BigQuery through the tool 'query_public_dataset'.

SQL RULES:
- ALWAYS use SUM(number) and GROUP BY name to get totals.
- ALWAYS wrap table names in backticks: `bigquery-public-data.usa_names.usa_1910_2013`.
- If the tool returns an error, report the error verbatim and do NOT answer with numbers from memory.
"""


def query_public_dataset(sql_query: str) -> dict:
    """Run a read-only SQL query against BigQuery public datasets and return up to 20 rows.

    Args:
        sql_query: A BigQuery Standard SQL SELECT statement.
    """
    from google.cloud import bigquery  # erst hier importieren: im Pickle steckt nur die Funktion

    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE"]
    if any(cmd in sql_query.upper() for cmd in forbidden):
        return {"error": "Modification queries are not allowed. Read-only access."}
    try:
        bq = bigquery.Client(project=PROJECT_ID)  # der Job läuft im Projekt deltorobarba: dafür braucht es jobUser
        job = bq.query(sql_query)
        rows = [dict(row) for row in job.result(max_results=20)]
        return {"rows": rows}  # wer die Abfrage ausgeführt hat, zeigt UC4g
    except Exception as e:
        return {"error": f"BigQuery: {e}"}


def build_identity_agent() -> LlmAgent:
    """Jedes Mal frisch bauen: Das lokal getestete Objekt ist ein anderes als das deployte."""
    return LlmAgent(
        name="agent_identity",
        model=MODEL,
        description="Agent with its own Agent Identity and a BigQuery tool (least privilege).",
        instruction=ANALYST_INSTRUCTION,
        tools=[query_public_dataset],
    )


print("✅ build_identity_agent() definiert, Tool: query_public_dataset")

✅ build_identity_agent() definiert, Tool: query_public_dataset


In [ ]:
# Lokaler Test (als du)
############################################
# Läuft mit DEINEN Credentials: BigQuery sieht hier dein Google-Konto als Aufrufer.

UC4_QUESTION = "What were the top 3 most popular female names in the USA in 1980?"


async def ask_with_tools(app, message: str, user_id: str = "deltorobarba"):
    """Zeigt Tool-Aufruf, Tool-Antwort (gekürzt) und Antwort des Agents; gleich für lokal und deployed."""
    session = await app.async_create_session(user_id=user_id)
    async for event in app.async_stream_query(user_id=user_id, session_id=session["id"], message=message):
        if event.get("error_message"):
            print(f"❌ [{event.get('author')}] {event.get('error_code')}: {event['error_message']}")
        for part in event.get("content", {}).get("parts", []):
            if "function_call" in part:
                print(f"🔧 Tool-Aufruf: {part['function_call']['name']}")
            elif "function_response" in part:
                print(f"📦 Tool-Antwort: {str(part['function_response'].get('response'))[:400]}")
            elif "text" in part and not part.get("thought"):
                print(f"🕵️ {part['text']}")


await ask_with_tools(AdkApp(agent=build_identity_agent()), UC4_QUESTION)

🔧 Tool-Aufruf: query_public_dataset
📦 Tool-Antwort: {'rows': [{'name': 'Jennifer', 'total': 58385}, {'name': 'Amanda', 'total': 35818}, {'name': 'Jessica', 'total': 33920}]}
🕵️ The top 3 most popular female names in the USA in 1980 were:

1. **Jennifer** – 58,385 babies
2. **Amanda** – 35,818 babies
3. **Jessica** – 33,920 babies


In [ ]:
# Agent mit Tool deployen (⚠️ --> once only!)
############################################
# update statt create: Die Engine und damit ihre Agent Identity bleiben dieselben, nur der Code ändert sich.
# DEPLOY_CONFIG kommt aus der Zelle "Deploy Agent with Agent Identity" (dort einmal ausführen, sie legt nichts neu an).

UC4_CONFIG = {
    **DEPLOY_CONFIG,
    "description": "Agent with Agent Identity and a BigQuery tool (least privilege)",
    # Für das Tool zusätzlich der BigQuery-Client, in derselben Version wie lokal
    "requirements": DEPLOY_CONFIG["requirements"] + [f"google-cloud-bigquery=={version('google-cloud-bigquery')}"],
}

client.agent_engines.update(name=RESOURCE_NAME, agent=AdkApp(agent=build_identity_agent()), config=UC4_CONFIG)
print("✅ Engine aktualisiert:", RESOURCE_NAME)
print("   Identität unverändert:", client.agent_engines.get(name=RESOURCE_NAME).api_resource.spec.effective_identity == AGENT_IDENTITY)

Identified the following requirements: {'google-cloud-aiplatform': '2.6.0', 'cloudpickle': '3.0.0', 'pydantic': '2.13.4'}
The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==2.6.0', 'google-adk==2.13.0', 'google-genai==2.33.0', 'cloudpickle==3.0.0', 'pydantic==2.13.4', 'google-cloud-bigquery==3.45.0']
Using bucket deltorobarba-agent-staging
Wrote to gs://deltorobarba-agent-staging/agent_governance/agent_identity/agent_engine.pkl
Writing to gs://deltorobarba-agent-staging/agent_governance/agent_identity/requirements.txt
Creating in-memory tarfile of extra_packages
Writing to gs://deltorobarba-agent-staging/agent_governance/agent_identity/dependencies.tar.gz
Using agent framework: google-adk
Update Agent Engine backing LRO: projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888/operations/7673859106175713280
View progress and logs at https://console.cloud.google.com/logs/query?project=deltorobarba&query=resource.type%3D%22aiplatform.goo

✅ Engine aktualisiert: projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888
   Identität unverändert: True


In [ ]:
# Test des deployten Agents OHNE BigQuery-Recht
############################################
# Erwartet: Das Tool meldet 403 "bigquery.jobs.create", und der Agent nennt den Fehler statt Zahlen.
# Die Standardrechte der Agent Identity (agentDefaultAccess) enthalten kein BigQuery.

await ask_with_tools(client.agent_engines.get(name=RESOURCE_NAME), UC4_QUESTION)

🔧 Tool-Aufruf: query_public_dataset
📦 Tool-Antwort: {'error': 'BigQuery: 403 POST https://bigquery.mtls.googleapis.com/bigquery/v2/projects/deltorobarba/jobs?prettyPrint=false: Access Denied: Project deltorobarba: User does not have bigquery.jobs.create permission in project deltorobarba.\n\nLocation: None\nJob ID: ebfde0d6-e12f-4b0c-8c2e-7ec062fff459\n'}
🕵️ An error occurred while executing the query:

```
BigQuery: 403 POST https://bigquery.mtls.googleapis.com/bigquery/v2/projects/deltorobarba/jobs?prettyPrint=false: Access Denied: Project deltorobarba: User does not have bigquery.jobs.create permission in project deltorobarba.

Location: None
Job ID: ebfde0d6-e12f-4b0c-8c2e-7ec062fff459
```


In [17]:
# BigQuery-Recht NUR für diese Agent Identity (⚠️ --> once only!)
############################################
# Member ist der Principal dieser einen Engine, nicht der geteilte Service Agent und kein principalSet.
# jobUser ist eine Projekt-Rolle, weil BigQuery-Jobs im Projekt laufen. Öffentliche Datasets darf jedes
# angemeldete Konto lesen; bei eigenen Daten käme dataViewer nur auf das eine Dataset dazu.

!gcloud projects add-iam-policy-binding {PROJECT_ID} --member="{AGENT_PRINCIPAL}" --role="roles/bigquery.jobUser" --condition=None --quiet > /dev/null && echo "✅ roles/bigquery.jobUser für {AGENT_PRINCIPAL}"
print("IAM-Propagation dauert ein bis zwei Minuten.")

Updated IAM policy for project [deltorobarba].
✅ roles/bigquery.jobUser für principal://agents.global.proj-622694106880.system.id.goog/resources/aiplatform/projects/622694106880/locations/us-central1/reasoningEngines/3345367756479397888
IAM-Propagation dauert ein bis zwei Minuten.


In [18]:
# Test des deployten Agents MIT BigQuery-Recht
############################################
# Erwartet: echte Zahlen (Jennifer 58,385 · Amanda 35,818 · Jessica 33,920); als wer, zeigt UC4g.
# Noch 403? Ein bis zwei Minuten warten und erneut ausführen.

await ask_with_tools(client.agent_engines.get(name=RESOURCE_NAME), UC4_QUESTION)

🔧 Tool-Aufruf: query_public_dataset
📦 Tool-Antwort: {'rows': [{'name': 'Jennifer', 'total': 58385}, {'name': 'Amanda', 'total': 35818}, {'name': 'Jessica', 'total': 33920}]}
🕵️ The top 3 most popular female names in the USA in 1980 were:

1. **Jennifer** – 58,385 births
2. **Amanda** – 35,818 births
3. **Jessica** – 33,920 births


In [ ]:
# Audit: Wer hat BigQuery abgefragt?
############################################
# BigQuery speichert zu jedem Job den Aufrufer (principal_subject). Hier stehen die drei Wege nebeneinander:
#   principal://…/reasoningEngines/…       der deployte Agent mit Agent Identity (UC4f)
#   user:alexmoheit@gmail.com              du, lokaler Test (UC4b)
#   serviceAccount:bq-mcp-analyst@…        der MCP-Server aus mcp.ipynb (UC1), egal welcher Agent ihn aufruft
# Der abgelehnte Versuch aus UC4d fehlt: Ohne Recht legt BigQuery gar keinen Job an.
# Public Datasets liegen in der Multi-Region US, deshalb `region-us`. Liest alle Jobs des Projekts (als Owner erlaubt).

AUDIT_SQL = """
SELECT creation_time, principal_subject, state
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND statement_type = 'SELECT'
ORDER BY creation_time DESC
LIMIT 10
"""

for job in bigquery.Client(project=PROJECT_ID).query(AUDIT_SQL).result():
    print(f"{job.creation_time:%Y-%m-%d %H:%M}  {job.state:5}  {job.principal_subject}")

In [ ]:
# (Optional) Aufräumen
############################################
# Löscht nur die Ressourcen dieses Notebooks; agents.ipynb, mcp.ipynb und gateway.ipynb bleiben unberührt.
# IAM-Bindings auf gelöschte principal://-Identitäten bleiben sonst stehen.

# client.agent_engines.delete(name=RESOURCE_NAME, force=True)
# for role in ["roles/aiplatform.expressUser", "roles/serviceusage.serviceUsageConsumer", "roles/browser",
#              "roles/bigquery.jobUser"]:
#     !gcloud projects remove-iam-policy-binding {PROJECT_ID} --member="{AGENT_PRINCIPAL}" --role="{role}" --condition=None --quiet > /dev/null
# !gcloud storage rm -r {STAGING_BUCKET}/{GCS_DIR}/agent_identity

*Troubleshooting*

| Symptom | Ursache / Fix |
|---|---|
| `identity_type` unbekannt oder ignoriert | Client ohne `http_options=dict(api_version="v1beta1")` (Setup) |
| 403 `aiplatform.sessions.create` oder `aiplatform.endpoints.predict` im Test, auch lange nach dem Deploy | Die Identität hat keine wirksamen Rechte, obwohl Google `agentDefaultAccess` vergibt. Beobachtet bis 2026-10-26, als das Projekt einer Organisation zugeordnet war (Trust-Domain `org-<Org-ID>`): die Ursache lag oberhalb des Projekts. Prüfen in *IAM & Admin › Policy Troubleshooter* mit Principal = `AGENT_PRINCIPAL`, Ressource = `//aiplatform.googleapis.com/` + `RESOURCE_NAME`, Berechtigung = `aiplatform.sessions.create` |
| 401 `Context-Aware Access requirements are not met` | Token der Agent Identity außerhalb der Runtime benutzt – gewollt, das Token ist an die Runtime gebunden |
| Tool meldet 403 `bigquery.jobs.create` | UC4e noch nicht ausgeführt, oder das Recht wirkt noch nicht (ein bis zwei Minuten warten) |
| Cloud Run antwortet dem Agent mit 401 „The access token could not be verified“ | Das ID-Token der Agent Identity stammt von STS, nicht von `accounts.google.com`; Cloud Run nimmt es nicht an, auch mit `run.invoker` (siehe UC4, Stand 2026-10-26) |
| Agent nennt Zahlen, obwohl kein Tool lief | Ein `McpToolset` konnte keine Verbindung aufbauen, und seine Tools fehlen still; Cloud-Run-Logs prüfen. Funktions-Tools geben Fehler als Text zurück |